# Control integral (Caveats)

*(Traducción a Python del livescript `c_integral_multi_estado.mlx`, usando `numpy`, `scipy` y `control`)*

Construimos las matrices del ejemplo del motor,

$$A = \begin{pmatrix}0&1\\0&-2\end{pmatrix}, \quad B = \begin{pmatrix}0\\1\end{pmatrix}$$

In [1]:
import numpy as np
import control as ct
from scipy.linalg import orth, null_space
import warnings

np.set_printoptions(precision=4, suppress=True)

In [2]:
A = np.array([[0, 1], [0, -2]])
B = np.array([[0], [1]])

Estudiamos la controlabilidad del sistema original,

In [3]:
Rc = np.linalg.matrix_rank(ct.ctrb(A, B))
Rc

np.int64(2)

El sistema original es controlable. Vamos de todas maneras a obtener autovectores,

In [4]:
l, v = np.linalg.eig(A.T)
Testv = B.T @ v
print('l =', l)
print('v =\n', v)
print('Testv =', Testv)

l = [-2.  0.]
v =
 [[0.     0.8944]
 [1.     0.4472]]
Testv = [[1.     0.4472]]


Queremos añadir al sistema control integral. Por las características del sistema, solo es posible controlar una de las dos salidas (pensad en el motor: puedo llevarlo a una posición fija, pero entonces su velocidad será 0. Puedo hacerlo girar con una determinada velocidad, pero entonces su posición crecerá indefinidamente).

## Caso 1: controlamos solo la posición

Supongamos que elegimos controlar la posición, es decir, hacer que llegue a un valor predefinido. Para ello necesitamos añadir control integral. Debemos analizar la estabilidad del par ampliado,

$$\begin{pmatrix}A & 0\\ C_2 & 0\end{pmatrix}, \begin{pmatrix}B \\ 0\end{pmatrix}$$

Construimos la ampliada para el caso particular en que solo tenemos como salida la posición y es la única variable de estado que queremos llevar a valor prefijado, $C_2 = (1\ 0)$.

In [5]:
A1 = np.block([
    [A,                np.zeros((2, 1))],
    [np.array([[1,0]]), np.zeros((1, 1))]
])
B1 = np.vstack([B, [[0]]])
print('A1 =\n', A1)
print('B1 =\n', B1)

A1 =
 [[ 0.  1.  0.]
 [ 0. -2.  0.]
 [ 1.  0.  0.]]
B1 =
 [[0]
 [1]
 [0]]


Vamos a pasar varios tests de controlabilidad,

In [6]:
Rc1 = np.linalg.matrix_rank(ct.ctrb(A1, B1))   # rango de la matriz de controlabilidad
Rc1

np.int64(3)

In [7]:
l1, v1 = np.linalg.eig(A1.T)
Testv1 = B1.T @ v1   # test de autovectores
print('l1 =', l1)
print('Testv1 =', Testv1)

l1 = [-2.  0.  0.]
Testv1 = [[ 1.      0.4472 -0.4472]]


Todos los valores del test son distintos de cero, y el rango de la matriz de controlabilidad coincide con la dimensión del sistema (3): la ampliada **sí es controlable**.

Comprobamos ahora si $A_1$ es Hurwitz mediante el test de Lyapunov. No debería pasarlo, ya que $A_1$ tiene un autovalor en el origen (no es Hurwitz).

**Nota importante para quien venga de MATLAB:** en MATLAB, `lyap(A1,-B1*B1')` lanza un **error explícito y capturable** con `try/catch` cuando la ecuación no tiene solución única (autovalores cuya suma es cero). En Python, `control.lyap` (que usa `scipy.linalg.solve_continuous_lyapunov` por debajo) **no lanza una excepción**: en su lugar, emite un `RuntimeWarning` y devuelve una "solución" numéricamente basura (típicamente con valores absurdamente grandes). Por eso, en vez de un `try/except`, comprobamos explícitamente el residuo de la ecuación para saber si la solución es de fiar.

In [8]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    P = ct.lyap(A1, -B1 @ B1.T)
    for aviso in w:
        print('AVISO:', aviso.category.__name__, '-', aviso.message)

residuo = A1 @ P + P @ A1.T - (-B1 @ B1.T)
print('norma del residuo (deberia ser ~0 si P es una solucion valida):',
      np.linalg.norm(residuo))

AVISO: RuntimeWarning - Input "a" has an eigenvalue pair whose sum is very close to or exactly zero. The solution is obtained via perturbing the coefficients.
norma del residuo (deberia ser ~0 si P es una solucion valida): 2.5353012004564585e+30


La norma del residuo es enorme: **no** hay una $P$ válida, tal y como esperábamos — $A_1$ no es Hurwitz, así que el test de Lyapunov falla (aunque en Python lo hayamos tenido que confirmar mirando el residuo, en vez de con una excepción limpia como en MATLAB).

## Caso 2: intentamos controlar ambos estados (imposible)

Vamos al caso en que tratamos de estabilizar ambos estados. Ahora $C_2 = I_{2\times2}$.

In [9]:
A2 = np.block([
    [A,          np.zeros((2, 2))],
    [np.eye(2),  np.zeros((2, 2))]
])
B2 = np.vstack([B, np.zeros((2, 1))])
print('A2 =\n', A2)
print('B2 =\n', B2)

A2 =
 [[ 0.  1.  0.  0.]
 [ 0. -2.  0.  0.]
 [ 1.  0.  0.  0.]
 [ 0.  1.  0.  0.]]
B2 =
 [[0.]
 [1.]
 [0.]
 [0.]]


Repetimos todos los tests,

In [10]:
Rc2 = np.linalg.matrix_rank(ct.ctrb(A2, B2))   # rango de la matriz de controlabilidad
Rc2

np.int64(3)

Da una matriz de rango 3: el sistema **no** es controlable. Pero si esto es cierto, entonces no cabe afirmar que si $(A,B)$ es un par controlable, también lo va a ser el de sus ampliadas, en general.

In [11]:
l2, v2 = np.linalg.eig(A2.T)
Testv2 = B2.T @ v2   # test de autovectores
print('l2 =', l2)
print('Testv2 =', Testv2)

l2 = [-2.  0.  0.  0.]
Testv2 = [[ 1.      0.4472 -0.4472  0.4472]]


Aparentemente pasa el test de autovectores (todos los valores son distintos de cero)... En realidad, **no**. El segundo y tercer autovalor son ambos cero — hay un autovalor cero repetido con **multiplicidad geométrica 2**, y el test de autovectores tal cual, columna a columna, no es válido en ese caso (como ya nos advertía la nota sobre autovalores repetidos en el notebook de controlabilidad).

> **Nota sobre el orden de los autovectores:** `numpy` no garantiza el mismo orden de autovectores que MATLAB para autovalores repetidos; si al ejecutar esta celda ves las columnas de `v2` en otro orden, identifica tú mismo cuáles corresponden a los dos autovalores cero repetidos, y ajusta los índices de la siguiente celda en consecuencia.

Los autovectores asociados al autovalor $0$ expanden un subespacio de dimensión 2, y **dentro de ese subespacio** sí hay un vector en el kernel de $B_2^T$. En concreto, para los autovectores que obtenemos aquí (columnas de índice 1 y 3, en indexado desde 0), la combinación $v = v_2[:,1] - v_2[:,3]$ es a la vez autovector de $A_2^T$ con autovalor 0 y está en el kernel de $B_2^T$:

In [12]:
v = v2[:, 1] - v2[:, 3]
print('v =', v)
print('A2.T @ v =', A2.T @ v, ' (deberia ser ~0: autovector de autovalor 0)')
print('B2.T @ v =', B2.T @ v, ' (deberia ser ~0: esta en el kernel de B2.T)')

v = [ 0.8944  0.      0.     -0.8944]
A2.T @ v = [0. 0. 0. 0.]  (deberia ser ~0: autovector de autovalor 0)
B2.T @ v = [0.]  (deberia ser ~0: esta en el kernel de B2.T)


Así que el sistema **es incontrolable**.

In [13]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    P = ct.lyap(A2, -B2 @ B2.T)
    for aviso in w:
        print('AVISO:', aviso.category.__name__, '-', aviso.message)

residuo = A2 @ P + P @ A2.T - (-B2 @ B2.T)
print('norma del residuo:', np.linalg.norm(residuo))

AVISO: RuntimeWarning - Input "a" has an eigenvalue pair whose sum is very close to or exactly zero. The solution is obtained via perturbing the coefficients.
norma del residuo: 2.5353012004564585e+30


De nuevo, un residuo enorme: no hay solución válida, tal y como esperábamos.

## Descomposición controlable de $A_2$

Añadimos una $C_2$ para obtener una transformación controlable. MATLAB pide que incluyamos esta $C$ para `ctrbf`, pero no la necesita realmente para el cálculo de la transformación — y en nuestra implementación con `orth`/`null_space` (ver el notebook de `estabilizabilidad.ipynb`) directamente no hace falta pasarla.

In [14]:
C2mat = ct.ctrb(A2, B2)
Qc = orth(C2mat)          # base ortonormal del subespacio controlable
Qu = null_space(Qc.T)     # complemento ortogonal (subespacio no controlable)
T = np.vstack([Qu.T, Qc.T])

A2c = T @ A2 @ T.T
B2c = T @ B2
print('A2c =\n', A2c)
print('B2c =\n', B2c)

A2c =
 [[-0.      0.     -0.      0.    ]
 [ 0.1345 -2.0084  1.2763 -0.3722]
 [ 0.3809 -0.0153 -0.3783  0.2519]
 [ 0.5803 -0.0103 -0.5854  0.3867]]
B2c =
 [[ 0.    ]
 [-0.8036]
 [ 0.5656]
 [-0.185 ]]


Por inspección de los resultados, podemos extraer la parte controlable y la no controlable.

In [15]:
nu = Qu.shape[1]   # numero de estados no controlables

A2cc = A2c[nu:, nu:]   # parte controlable
B2cc = B2c[nu:, :]
Rc2cc = np.linalg.matrix_rank(ct.ctrb(A2cc, B2cc))

A2cu = A2c[:nu, :nu]   # parte NO controlable

print('A2cc =\n', A2cc)
print('B2cc =\n', B2cc)
print('Rc2cc =', Rc2cc)
print('A2cu =', A2cu)

A2cc =
 [[-2.0084  1.2763 -0.3722]
 [-0.0153 -0.3783  0.2519]
 [-0.0103 -0.5854  0.3867]]
B2cc =
 [[-0.8036]
 [ 0.5656]
 [-0.185 ]]
Rc2cc = 3
A2cu = [[-0.]]


La matriz no controlable tiene un único autovalor cero,

In [16]:
np.linalg.eigvals(A2cu)

array([-0.])

por lo que el sistema tampoco sería **estabilizable** (autovalor en el eje imaginario dentro de la parte no controlable): confirma, desde otro ángulo, la misma conclusión que obtuvimos con el test de autovectores — intentar forzar control integral simultáneo sobre posición y velocidad en este motor no es posible.